<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/06_foundation_models/notebooks/sesion_06_6_integrador_estilo_llm.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/06_foundation_models/notebooks/sesion_06_6_integrador_estilo_llm.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F06_foundation_models%2Fnotebooks%2Fsesion_06_6_integrador_estilo_llm.ipynb)

# SI7011 · Sesión 06 · Ejercicio integrador: ¿hay que ajustar un LLM para que responda con un estilo?
**Juan David Martínez Vargas · Universidad EAFIT**

En el notebook 6.3 convertiste un modelo **base** en uno que responde. Aquí partimos de un modelo que **ya conversa**, Qwen2.5-0.5B-Instruct, y queremos que responda en el castellano de *Don Quijote*. La pregunta: **¿basta con pedírselo en el prompt, o hay que ajustarlo?** Lo haces como en la práctica: datos en el formato de chat de TRL, `SFTTrainer` con LoRA de peft, y MLflow.

| Etapa | Qué haces | Artefacto |
|---|---|---|
| Datos | unas 380 preguntas de Dolly con su respuesta reescrita en el estilo: 40 para validar, unas 60 de test y el resto para entrenar (listo) | ficha de datos en MLflow |
| Formato | cada fila como conversación *prompt / completion* (**TODO 1**) | |
| Ajuste | `LoraConfig` y `SFTConfig` (**TODO 2**); `SFTTrainer` registra en MLflow (listo) | `lora_estilo/` |
| Evaluación | $J$ de test para base, base con el estilo en el prompt y LoRA (**TODO 3**); un juez simple del estilo (listo) | tablas en MLflow |
| Despliegue | `responder_raw`: de una pregunta en texto a una respuesta, con contrato (**TODO 4**) | |

**Usa GPU** (Kaggle T4, unos 10 minutos) con **Internet** activado (el modelo viene de Hugging Face y TRL se instala con `pip`). **Datos:** en Kaggle agrega el dataset [`juanmartinezv4399/si7011-s06-dolly-quijote`](https://www.kaggle.com/datasets/juanmartinezv4399/si7011-s06-dolly-quijote) con **Add Input**; en Colab se baja con tu secreto `KAGGLE_API_TOKEN`.

<span style="font-size:smaller">Adaptado de MIT 6.S191 *Introduction to Deep Learning*, Lab 3 (Copyright (c) 2025 MIT Introduction to Deep Learning, licencia MIT). Las preguntas son de Databricks Dolly-15k en español (CC BY-SA 3.0); las respuestas con estilo se generaron con Qwen2.5-7B-Instruct.</span>

In [ ]:
%pip install -q mlflow "trl==1.16.0" peft

In [ ]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')     # Kaggle "GPU T4 x2": una sola GPU (SFTTrainer no admite DataParallel)
# Kaggle trae torchao 0.10, que las versiones recientes de transformers y peft rechazan al importarse
# ("Found an incompatible version of torchao"). Este notebook no lo usa: si es viejo, se desinstala.
import os
from importlib.metadata import version, PackageNotFoundError
try:
    if tuple(int(x) for x in version('torchao').split('.')[:2]) < (0, 16):
        os.system('pip uninstall -y -q torchao')
except (PackageNotFoundError, ValueError):
    pass

import hashlib, json, math, random, textwrap, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers, trl, peft
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')
import mlflow
mlflow.set_tracking_uri('sqlite:///mlflow.db')
os.environ['MLFLOW_TRACKING_URI'] = 'sqlite:///mlflow.db'       # el que usa el Trainer de Hugging Face
os.environ['MLFLOW_EXPERIMENT_NAME'] = 'S06-estilo-llm'
mlflow.set_experiment('S06-estilo-llm')

random.seed(0); torch.manual_seed(0)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '· trl', trl.__version__, '· peft', peft.__version__,
      '· MLflow', mlflow.__version__, '·', dev)

## 1. Datos (listo)
Cada fila: una pregunta de Dolly en español, la respuesta original (`response`) y la misma respuesta reescrita en el estilo (`response_estilo`). Son preguntas **distintas** de las del notebook 6.3. Del `train` separamos 40 para validar, y de `test` quitamos las preguntas que también están en `train`; `test` se usa solo en el TODO 3. La última línea mide cuántas respuestas de entrenamiento usan algún giro de la época: lo que el modelo tiene para aprender.

In [ ]:
KAGGLE_HANDLE = 'juanmartinezv4399/si7011-s06-dolly-quijote'
ROOTS = [Path('.'), Path('data'), Path('/kaggle/input'), Path('/teamspace/studios/this_studio/data')]

def kaggle_token():
    # Colab: secreto KAGGLE_API_TOKEN. Local o Lightning: variable de entorno o ~/.kaggle/access_token.
    if 'KAGGLE_API_TOKEN' not in os.environ:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
        except Exception:
            pass
    return 'KAGGLE_API_TOKEN' in os.environ or (Path.home() / '.kaggle' / 'access_token').exists()

def buscar(roots, patron='dolly_quijote_es.csv'):
    for r in [r for r in roots if r.exists()]:
        hits = sorted(r.rglob(patron))
        if hits:
            return hits[0]
    return None

ruta = buscar(ROOTS)
if ruta is None and kaggle_token():
    import kagglehub                                        # preinstalado en Colab y Kaggle
    ruta = buscar([Path(kagglehub.dataset_download(KAGGLE_HANDLE))])
assert ruta is not None, f'No encontré dolly_quijote_es.csv: en Kaggle agrega {KAGGLE_HANDLE} con Add Input.'
df = pd.read_csv(ruta)
train_all, test = df[df.split == 'train'].reset_index(drop=True), df[df.split == 'test'].reset_index(drop=True)
test = test[~test.instruction.isin(train_all.instruction)].reset_index(drop=True)   # ninguna pregunta de test se vio al entrenar
val, train = train_all.iloc[:40].reset_index(drop=True), train_all.iloc[40:].reset_index(drop=True)
if dev == 'cpu':
    train, val, test = train.iloc[:40], val.iloc[:12], test.iloc[:12]
data_card = {'fuente': str(ruta), 'sha256': hashlib.sha256(ruta.read_bytes()).hexdigest()[:16],
             'train': len(train), 'val': len(val), 'test': len(test)}
print(data_card)
GIROS = ['vuestra merced', 'agora', 'mesmo', 'asaz', 'os digo', 'pardiez', 'aqueste', 'hube de', 'sabed', 'en verdad']
con_giros = lambda r: sum(g in r.lower() for g in GIROS) >= 1      # un juez muy simple del estilo
print(f"respuestas de entrenamiento con algún giro de la época: {train.response_estilo.apply(con_giros).mean():.0%} "
      f"(Dolly original: {train.response.apply(con_giros).mean():.0%})")
e = train.iloc[0]
print('\n▶', e.instruction, '\n  Dolly:  ', e.response, '\n  estilo: ', e.response_estilo)

## 2. El modelo (listo)
`mensajes` arma la conversación (con o sin mensaje de sistema) y `responder` aplica la **plantilla de chat del modelo** (`apply_chat_template`) y genera con greedy hasta `<|im_end|>`. Mira la plantilla, y cómo responde el modelo con y sin el estilo en el sistema.

In [ ]:
MODELO = 'Qwen/Qwen2.5-0.5B-Instruct'      # la versión que ya conversa del modelo base del notebook 6.3
tok = AutoTokenizer.from_pretrained(MODELO)
modelo = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32).to(dev)
FIN = tok.convert_tokens_to_ids('<|im_end|>')               # el fin de turno de la plantilla de Qwen
ESTILO = ('Responde siempre en el castellano de Don Quijote de la Mancha (principios del siglo XVII): '
          'vuestra merced, agora, mesmo, asaz, os digo; tono cortés y algo grandilocuente.')

def mensajes(pregunta, sistema=None):
    return ([{'role': 'system', 'content': sistema}] if sistema else []) + [{'role': 'user', 'content': pregunta}]

@torch.no_grad()
def responder(m, pregunta, sistema=None, max_nuevos=150):
    x = tok.apply_chat_template(mensajes(pregunta, sistema), add_generation_prompt=True, return_tensors='pt',
                                return_dict=True).to(dev)
    out = m.generate(**x, max_new_tokens=max_nuevos, do_sample=False, eos_token_id=FIN,
                     pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, x['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print(tok.apply_chat_template(mensajes('¿Qué es un átomo?'), add_generation_prompt=True, tokenize=False))
for q in test.instruction[:2]:
    print('▶', q)
    print('  base:          ', textwrap.shorten(responder(modelo, q), 300))
    print('  base + estilo: ', textwrap.shorten(responder(modelo, q, ESTILO), 300))

## 3. TODO 1 · Cada fila como conversación · 5 min
TRL entrena con conversaciones en el formato *prompt / completion*: el prompt es lo que el modelo lee, la completion lo que tiene que aprender a escribir. Escribe `a_conversacion(fila, sistema=None)` que devuelva

```python
{'prompt': mensajes(pregunta, sistema),
 'completion': [{'role': 'assistant', 'content': <la respuesta CON estilo>}]}
```

`conjunto` (listo) convierte una tabla entera en un `Dataset` de Hugging Face.

In [ ]:
def a_conversacion(fila, sistema=None):
    # TODO 1: {'prompt': [mensajes], 'completion': [mensaje del asistente con la respuesta con estilo]}
    raise NotImplementedError('Completa TODO 1')

In [ ]:
def conjunto(filas, sistema=None):
    """Un Dataset de Hugging Face en el formato prompt / completion conversacional de TRL."""
    return Dataset.from_list([a_conversacion(f, sistema) for f in filas.to_dict('records')])

ej = a_conversacion(train.iloc[0].to_dict())
assert set(ej) == {'prompt', 'completion'}, "Esperaba las claves 'prompt' y 'completion'."
assert ej['prompt'][-1] == {'role': 'user', 'content': train.instruction[0]}, 'El prompt termina con el mensaje del usuario.'
assert ej['completion'] == [{'role': 'assistant', 'content': train.response_estilo[0]}], \
    'La completion es un solo mensaje del asistente con la respuesta CON estilo.'
ej_s = a_conversacion(train.iloc[0].to_dict(), ESTILO)
assert ej_s['prompt'][0] == {'role': 'system', 'content': ESTILO} and len(ej_s['prompt']) == 2, \
    'Con sistema, el prompt empieza con el mensaje de sistema.'
D_train, D_val = conjunto(train), conjunto(val)
print('a_conversacion correcta ·', len(D_train), 'conversaciones de entrenamiento')
print(D_train[0])

## 4. TODO 2 · LoRA y el entrenamiento supervisado · 10 min
Define dos objetos (mira el notebook 6.3, sección 5):
- `lora = LoraConfig(...)`: $r = 16$, `lora_alpha=32`, `lora_dropout=0.0`, `task_type='CAUSAL_LM'`, sobre `q_proj`, `k_proj`, `v_proj`, `o_proj`;
- `args = SFTConfig(...)`: `output_dir='sft_estilo'`, `run_name='1-lora-r16'`, la pérdida **solo en la respuesta** (`completion_only_loss=True`), `max_length=384`, dos épocas, lotes de 8 (entrenamiento y evaluación), `learning_rate=5e-4`, coseno con `warmup_steps=0.1`, `fp16` en la GPU y `bf16=False` (la T4 no tiene bfloat16), evaluación cada 10 pasos (`eval_strategy='steps'`, `eval_steps=10`), `logging_steps=5`, `save_strategy='no'` y `report_to='mlflow'`.

La celda de prueba crea el `SFTTrainer` y te muestra qué tokens cuentan en la pérdida: deben ser solo la respuesta y el fin de turno.

**Predice:** en el TODO 3 vas a medir $J$ sobre las respuestas con estilo de test en tres condiciones. ¿La del modelo con el estilo **pedido en el prompt** quedará más cerca de la del modelo base o de la de LoRA?

In [ ]:
# TODO 2: LoRA y el entrenamiento supervisado
lora = None     # LoraConfig(...)
args = None     # SFTConfig(...)

In [ ]:
assert lora is not None and args is not None, 'Completa TODO 2: define lora y args.'
assert lora.r == 16 and set(lora.target_modules) == {'q_proj', 'k_proj', 'v_proj', 'o_proj'}, \
    'LoRA con r = 16 en las cuatro matrices de la atención.'
assert args.completion_only_loss is True, 'La pérdida debe contar solo la respuesta (completion_only_loss).'
assert args.num_train_epochs == 2 and args.per_device_train_batch_size == 8 and args.learning_rate == 5e-4
assert args.eval_strategy == 'steps' and 'mlflow' in args.report_to, 'Evalúa cada pocos pasos y registra en MLflow.'
assert not args.bf16, 'La T4 no tiene bfloat16: usa fp16.'
trainer = SFTTrainer(model=modelo, args=args, train_dataset=D_train, eval_dataset=D_val,
                     processing_class=tok, peft_config=lora)
trainer.model.print_trainable_parameters()
lote = next(iter(trainer.get_train_dataloader()))
ids, etiquetas = lote['input_ids'][0], lote['labels'][0]
print('cuenta en la pérdida:', repr(tok.decode(ids[etiquetas != -100])))

## 5. Entrenar (listo)
`trainer.train()` registra en MLflow los hiperparámetros, la pérdida de entrenamiento y la de validación. Después guardamos **solo el adaptador** (las matrices $A_r$ y $B$, unos MB) con el **contrato**, y los subimos a la misma corrida.

In [ ]:
t0 = time.time()
trainer.train()
print(f'{time.time() - t0:.0f} s')
run_id = mlflow.last_active_run().info.run_id              # la corrida que abrió el Trainer
CONTRATO = {'modelo': MODELO, 'max_tokens_pregunta': 128, 'max_nuevos': 150, 'plantilla': 'chat de Qwen, sin sistema'}
trainer.model.save_pretrained('lora_estilo')               # solo el adaptador: unos MB
json.dump(CONTRATO, open('lora_estilo/contrato.json', 'w'), ensure_ascii=False, indent=1)
with mlflow.start_run(run_id=run_id):
    mlflow.log_params({f'data_{k}': v for k, v in data_card.items()})
    mlflow.log_artifacts('lora_estilo', 'lora_estilo')
print('adaptador y contrato en MLflow, corrida', run_id)

In [ ]:
def nll(datos):
    """La J de la respuesta sobre un Dataset, con el modelo como esté (con o sin el adaptador activo)."""
    ev = SFTTrainer(model=trainer.model, processing_class=tok, train_dataset=datos, eval_dataset=datos,
                    args=SFTConfig(output_dir='eval_tmp', per_device_eval_batch_size=8, max_length=384,
                                   fp16=dev == 'cuda', bf16=False, report_to='none'))
    return ev.evaluate()['eval_loss']

## 6. TODO 3 · ¿Basta el prompt? · 15 min
Escribe `comparar(filas)`: devuelve un diccionario con la $J$ de las respuestas con estilo de `filas` en tres condiciones (diapositiva 74):

| Clave | Adaptador | Mensaje de sistema |
|---|---|---|
| `'base'` | apagado (`with trainer.model.disable_adapter():`) | ninguno |
| `'base + estilo en el prompt'` | apagado | `ESTILO` |
| `'LoRA'` | activo | ninguno |

`nll(conjunto(filas, sistema))` ya calcula $J$ sobre un conjunto. La comparación es justa porque las tres condiciones usan el **mismo tokenizador** y la misma respuesta objetivo: solo cambia el prompt o los pesos.

In [ ]:
def comparar(filas):
    # TODO 3: {'base': J, 'base + estilo en el prompt': J, 'LoRA': J} sobre las respuestas con estilo de `filas`
    raise NotImplementedError('Completa TODO 3')

In [ ]:
r = comparar(test.iloc[:8])
assert set(r) == {'base', 'base + estilo en el prompt', 'LoRA'}, 'Esperaba las tres condiciones.'
with trainer.model.disable_adapter():
    assert abs(r['base'] - nll(conjunto(test.iloc[:8]))) < 1e-3, '"base" se mide con el adaptador apagado y sin sistema.'
assert abs(r['LoRA'] - nll(conjunto(test.iloc[:8]))) < 1e-3, '"LoRA" se mide con el adaptador activo y sin sistema.'
assert r['base + estilo en el prompt'] != r['base'], 'La condición con el estilo en el prompt usa sistema=ESTILO.'

tabla = pd.DataFrame([{'condición': k, 'J_test': v, 'perplejidad': math.exp(v)} for k, v in comparar(test).items()])
with mlflow.start_run(run_id=run_id):
    mlflow.log_table(tabla, 'comparacion_test.json')
print(tabla.round(3).to_string(index=False))

$J$ dice cuánto se parecen las respuestas de test a las que daría el modelo, pero no se lee fácil. Un **juez muy simple**: la fracción de 20 respuestas generadas que usan al menos un giro de la época (`GIROS`, la misma medida de la sección 1). También el largo, porque el ajuste cambia más que el estilo.

In [ ]:
N_GEN = 20 if dev == 'cuda' else 2
m = trainer.model
gen = {'base': [], 'base + estilo en el prompt': [], 'LoRA': []}
for q in test.instruction[:N_GEN]:
    with m.disable_adapter():
        gen['base'].append(responder(m, q)); gen['base + estilo en el prompt'].append(responder(m, q, ESTILO))
    gen['LoRA'].append(responder(m, q))
tabla['respuestas con giros de la época'] = [np.mean([con_giros(r) for r in gen[k]]) for k in tabla['condición']]
tabla['palabras por respuesta'] = [np.mean([len(r.split()) for r in gen[k]]) for k in tabla['condición']]
with mlflow.start_run(run_id=run_id):
    mlflow.log_table(tabla, 'comparacion_test_con_juez.json')
print(tabla.round(3).to_string(index=False), '\n')
for i in range(min(3, N_GEN)):
    print('▶', test.instruction[i])
    for k in gen:
        print(f'  {k + ":":<28}', textwrap.shorten(gen[k][i], 260))
    print()

## 7. Cambia una sola cosa · 10 min
Elige **un** cambio: copia la celda del TODO 2 con otro `run_name`, crea un `SFTTrainer` nuevo (con el modelo recién cargado: `AutoModelForCausalLM.from_pretrained(MODELO, ...)`), entrena y repite `comparar(test)`. Antes de correrlo, escribe tu predicción.
- $r = 4$ o $r = 64$;
- LoRA solo en `q_proj` y `v_proj`;
- cuatro épocas en lugar de dos (mira la curva de validación en MLflow);
- entrenar con la respuesta **original** de Dolly en lugar de la con estilo: ¿qué le pasa a la $J$ del estilo?

_Mi cambio y mi predicción:_

In [ ]:
# modelo_2 = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32).to(dev)
# ...

## 8. TODO 4 · Despliegue: `responder_raw` · 10 min
Escribe `responder_raw(pregunta, carpeta)`: recibe una pregunta en texto y la carpeta del adaptador (la que bajamos de MLflow), y devuelve la respuesta en texto. Usa solo lo que hay en la carpeta:
1. el **contrato** (`contrato.json`): la pregunta debe ser un `str` no vacío y tener como máximo `max_tokens_pregunta` tokens; si no, `ValueError`;
2. el modelo: carga el modelo base que dice el contrato y ponle el adaptador con `PeftModel.from_pretrained(base, carpeta)`. Guárdalo en `_CACHE` para no cargarlo en cada llamada;
3. la plantilla **sin** sistema: `responder(modelo, pregunta, max_nuevos=...)` con el `max_nuevos` del contrato.

In [ ]:
_CACHE = {}

def responder_raw(pregunta, carpeta='lora_estilo'):
    # TODO 4: valida el contrato, arma base + adaptador (una vez, en _CACHE) y devuelve la respuesta en texto
    raise NotImplementedError('Completa TODO 4')

In [ ]:
carpeta = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path='lora_estilo')
for q in test.instruction[:3]:
    a, b = responder_raw(q, carpeta), responder(trainer.model, q)
    assert a == b, 'responder_raw debe dar lo mismo que el modelo entrenado (greedy, sin sistema).'
    print('1. Consistencia ✓ ·', textwrap.shorten(a, 150))
for malo in ['', '   ', None, 'palabra ' * 400]:
    try:
        responder_raw(malo, carpeta); raise AssertionError(f'Debió rechazar {str(malo)[:20]!r}.')
    except ValueError as e:
        print('2. Contrato: rechazada →', e)

## 9. Preguntas
1. ¿Se cumplió tu predicción? Con la tabla del TODO 3: ¿cuánto de la distancia entre "base" y "LoRA" cubre el prompt?
2. Mira las respuestas generadas y la columna del juez. ¿Coincide lo que ves con lo que dice $J$? Da un ejemplo en que no coincidan.
3. ¿Por qué no se puede comparar esta $J$ con la NLL del notebook 6.3 ni con la de otro modelo con otro tokenizador?
4. ¿Qué costó cada opción? El prompt: tokens de más en **cada** llamada. LoRA: el entrenamiento y una carpeta de unos MB. ¿Cuándo conviene cada una?
5. En producción llega una pregunta en inglés. ¿Qué haría tu `responder_raw`? ¿Qué agregarías al contrato?

_Respuestas:_

**Si quieres ir más allá:** el juez de giros es fácil de engañar (una respuesta que solo repite "vuestra merced" saca 1). MIT usa un LLM como juez; ¿qué problemas tendría aquí, y cómo comprobarías que el juez está de acuerdo con una persona?